# ViCare · Tra cứu kiến thức sức khỏe tiếng Việt

**Vương Đình Bình · AI Challenge 24h**

Một câu trả lời chỉ hữu ích khi đoạn dẫn nguồn nói đúng điều người dùng hỏi. ViCare tìm tài liệu trong ViHealthQA, kiểm tra xem bằng chứng có đủ và chỉ hiển thị câu trả lời khi vượt qua các bước kiểm tra. Câu có dấu hiệu cấp cứu được chuyển sang hướng dẫn khẩn cấp. Đây là demo học thuật, không dùng để chẩn đoán hoặc kê đơn. Chỉ nhập câu hỏi chung, không nhập tên, số điện thoại, địa chỉ hoặc mã hồ sơ.

**Câu hỏi → kiểm tra cấp cứu và phạm vi → E5 + Chroma / BM25 → xếp hạng tài liệu → kiểm tra bằng chứng → Qwen3 → kiểm tra bệnh cảnh và trích dẫn.**

Mở bằng Colab với một **runtime mới, miễn phí, Linux x86_64**, có thể chọn T4 nếu còn hạn mức. Chọn **Runtime → Run all**. Lần đầu phải tải dependencies, kho dữ liệu, E5, Ollama, Qwen và lập chỉ mục 11.227 đoạn. Giữ phiên đến khi xuất hiện file ZIP ở cuối. Các pha dùng Python riêng để giải phóng bộ nhớ khi tiến trình kết thúc. E5 chạy CPU ở mọi pha, Ollama dùng phần cứng sẵn có trong cùng VM. Bảng và câu trả lời chỉ xuất hiện sau khi chạy.


## 1. Môi trường và mã nguồn

Notebook sử dụng commit `b9a586aecf9837fdf991dddd55f07d26b43d3b64`. Ô dưới ghi tài nguyên thực tế của VM, lấy đúng commit và tách bộ kết quả của mỗi lượt chạy. Các ô có thể dùng lại cache trong cùng VM. Không cần Drive, token cá nhân hoặc truy cập máy của tác giả.


In [ ]:
import csv
import html
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import HTML, display

SOURCE = "b9a586aecf9837fdf991dddd55f07d26b43d3b64"
REMOTE = "https://github.com/Vdbinh05/ViCare-AI-Challenge.git"
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Chọn runtime Colab Linux x86_64.")
BASE = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "vicare-colab"
BASE.mkdir(parents=True, exist_ok=True)
SRC = BASE / ("source-" + SOURCE)
WORK = BASE / ("runtime-" + SOURCE[:12])
WORK.mkdir(exist_ok=True)
RUN = BASE / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
RUN.mkdir(parents=True)
PY = WORK / "venv/bin/python"
ENV = {k: v for k, v in os.environ.items()
       if not k.startswith(("VICARE_", "HF_", "OLLAMA_", "UV_", "PIP_"))
       and k not in ("HUGGING_FACE_HUB_TOKEN", "PYTHONPATH", "VIRTUAL_ENV")}
ENV.update(HF_HOME=str(BASE / "hf-cache"), HF_HUB_DISABLE_IMPLICIT_TOKEN="1",
           HF_HUB_DISABLE_TELEMETRY="1", ANONYMIZED_TELEMETRY="False",
           VICARE_INDEX_DIR=str(WORK / "chroma"), VICARE_LLM_MODEL="qwen3:4b-instruct",
           VICARE_OLLAMA_HOST="http://127.0.0.1:11434", VICARE_OLLAMA_TIMEOUT="180",
           VICARE_TORCH_THREADS=str(min(8, len(os.sched_getaffinity(0)))),
           OPENBLAS_NUM_THREADS="1",
           PYTHONUTF8="1", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")

def run(argv, *, capture=False, **kwargs):
    return subprocess.run([str(arg) for arg in argv], cwd=SRC,
                          env=ENV, check=True, text=True, encoding="utf-8",
                          stdout=subprocess.PIPE if capture else None, **kwargs)

def read(name):
    return json.loads((RUN / name).read_text(encoding="utf-8"))

def write(name, value):
    (RUN / name).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag}>{html.escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(v, "td") for v in rows) + "</table>"))

def gib(value):
    return round(value / 2**30, 2)

ram = {line.split(":")[0]: int(line.split()[1]) * 1024 for line in
       Path("/proc/meminfo").read_text().splitlines() if line.startswith(("MemTotal:", "MemAvailable:"))}
smi = shutil.which("nvidia-smi")
gpu = subprocess.run([smi, "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True) if smi else None
host = {"utc": datetime.now(timezone.utc).isoformat(), "source_revision": SOURCE,
        "kernel_python": platform.python_version(), "linux": platform.release(),
        "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG"),
        "cpu_threads": ENV["VICARE_TORCH_THREADS"], "e5_device": "cpu",
        "gpu_vram_driver": gpu.stdout.strip() if gpu and gpu.returncode == 0 else None,
        "gpu_probe_status": (gpu.returncode if gpu else "nvidia-smi absent"),
        "ram_gib": gib(ram["MemTotal"]), "ram_available_gib": gib(ram["MemAvailable"]),
        "disk_free_gib": gib(shutil.disk_usage(BASE).free), "ollama_timeout_s": 180}
write("environment.json", host)
print(json.dumps(host, ensure_ascii=False, indent=2))
if not SRC.exists():
    with tempfile.TemporaryDirectory(dir=BASE) as temp:
        stage = Path(temp) / "source"
        subprocess.run(["git", "init", "--quiet", stage], check=True)
        subprocess.run(["git", "-C", stage, "remote", "add", "origin", REMOTE], check=True)
        subprocess.run(["git", "-C", stage, "fetch", "--quiet", "--depth", "1", "origin", SOURCE], check=True)
        subprocess.run(["git", "-C", stage, "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
        stage.rename(SRC)
if (run(["git", "rev-parse", "HEAD"], capture=True).stdout.strip() != SOURCE
        or run(["git", "remote", "get-url", "origin"], capture=True).stdout.strip() != REMOTE
        or run(["git", "status", "--porcelain", "--untracked-files=normal"], capture=True).stdout.strip()):
    raise RuntimeError("Thư mục source khác commit sạch đã ghim. Chọn runtime mới.")
print("Source đã xác minh:", SOURCE)


## 2. Cài môi trường riêng và chạy kiểm thử hồi quy

Kernel Colab chỉ điều phối và hiển thị. Toàn bộ code của ViCare dùng CPython 3.11.14 trong môi trường tách biệt cùng danh sách dependency Linux đã khóa. Không cần khởi động lại kernel. Bộ kiểm thử bao gồm các ca an toàn và xử lý lỗi. Các phép đo với dữ liệu thật nằm ở các ô tiếp theo.


In [ ]:
run([sys.executable, "-m", "benchmarks.colab_runtime", "install", "--work", WORK])
versions = json.loads(run([PY, "-c", """
import json, platform, importlib.metadata as md
print(json.dumps({"python": platform.python_version(), "packages":
                  {d.metadata["Name"]: d.version for d in md.distributions()}}))
"""], capture=True).stdout)
write("versions.json", versions)
print("Python của ViCare", versions["python"], "· Torch", versions["packages"]["torch"])
regression = run([PY, "-m", "pytest", "-q"], capture=True).stdout
print(regression)
write("regression.json", {"command": "python -m pytest -q", "summary": regression.splitlines()[-1]})


## 3. Tái tạo corpus, Dev100 và chỉ mục

Pipeline sử dụng cả ba split để tạo kho tài liệu của bài tập. Nó ẩn điện thoại/email trong câu trả lời nguồn, hợp nhất văn bản trùng và giữ ID chuẩn. Đoạn dài tối đa 200 từ, chồng 40 từ. E5 nhận tiền tố `passage:` và `query:`. Xác minh checksum của 9.895 tài liệu, 11.227 đoạn và thứ tự 100 câu trước khi đo. Chỉ evaluator được đọc nhãn gold. Chỉ mục cùng chữ ký dùng lại được sau khi chạy lại ô hoặc gián đoạn trong cùng VM.


In [ ]:
run([PY, "-c", """
import json, sys, time
from pathlib import Path
from src.config import EMBEDDING_MODEL, EMBEDDING_REVISION, Settings
from src.dataset import dev_identity, load_vihealthqa, prepare_dataset
from src.retrieval import build_index
ds = load_vihealthqa()
corpus, chunks, dev = prepare_dataset(ds)
identity = dev_identity(ds, corpus, chunks, dev)
identity.pop("questions")
started = time.perf_counter()
manifest = build_index(Settings.from_env(), chunks)
summary = {**identity, "dev_count": len(dev), "embedding_model": EMBEDDING_MODEL,
           "embedding_revision": EMBEDDING_REVISION, "index_manifest": manifest,
           "build_or_reuse_seconds": round(time.perf_counter()-started, 2)}
Path(sys.argv[1]).write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
""", RUN / "corpus.json"])
print(json.dumps(read("corpus.json"), ensure_ascii=False, indent=2))


## 4. Vì sao chọn cấu hình này?

Bảng đầu đọc các artifact thí nghiệm đã ghi trên Windows ở đúng commit nguồn. Trên Train200, dùng cosine chính xác thay HNSW trên **cùng vector E5** đưa số câu có gold trong top 5 từ **123 lên 125/200**. Trên NoDiac50 cả hai tìm được **22/50**. Reranker đạt **119/200** và **11/50**. Vì vậy bản chọn giữ BM25/RRF và dùng cosine chính xác trên toàn bộ 11.227 vector trước khi lấy ứng viên. Không có đoạn nào vượt giới hạn 512 token của E5 trong phép đếm có tiền tố và special tokens, nên giữ nguyên chunking. Đây là kết quả trên tập chọn; Dev100 bên dưới được đo sau khi đã khóa.


In [ ]:
study = SRC / "results/colab/retrieval"
selection = json.loads((study / "selection.json").read_text(encoding="utf-8"))
previous = []
for split in ("train200", "nodiac50"):
    for filename in (f"{split}.json", f"{split}_rerank.json"):
        path = study / filename
        if path.is_file():
            historical = json.loads(path.read_text(encoding="utf-8"))
            for name, m in historical["metrics"].items():
                previous.append([split, name, m["n"], *(f'{m["recall"][f"@{k}"]:.1%}' for k in (1, 3, 5, 10)),
                                 f'{m["candidate50_document_coverage"]:.1%}', m["latency_ms"]["p95"]])
table(["Tập chọn · Windows", "Biến thể", "Số câu", "R@1", "R@3", "R@5", "R@10",
       "Gold trong candidate50", "P95 (ms)"], previous)
print("Đã khóa:", selection["selected"], "· UTC:", selection["locked_at_utc"])
for candidate, reasons in selection["ineligible_reasons"].items():
    print(candidate, "—", ", ".join(reasons) if reasons else "đạt tiêu chí")
explanations = {
    "exact": "Giữ nguyên E5, BM25 và RRF; dùng cosine chính xác trên toàn bộ vector trước khi lấy ứng viên.",
    "rerank": "Giữ tập 50 tài liệu ứng viên và chấm các đoạn trước khi xếp hạng lại theo tài liệu.",
    "locked": "Giữ Hybrid đầu kỳ vì các phương án mới chưa vượt tiêu chí đã công bố."}
print(explanations[selection["selected"]])


### Đo lại trên Dev100 trong runtime này

Dense tìm đoạn bằng E5/Chroma. Hybrid kết hợp danh sách ngữ nghĩa với BM25 theo tài liệu bằng RRF. Mỗi hạng là một **tài liệu**. Bảng hiển thị tỉ lệ có gold ở từng top K, trong toàn chỉ mục, trong hợp ứng viên và trong 50 tài liệu đầu thuộc nhóm ứng viên. Latency đo cho từng câu. Dev100 từng được dùng để phân tích lỗi. Train200/NoDiac50 từng dùng chọn cấu hình. Xem giao thức và quyết định chọn cấu hình trong source đã ghim.

`locked` là Hybrid đầu kỳ, `hybrid_v2` là biến thể HNSW độ sâu 80 và `selected` là Hybrid dùng cosine chính xác. Bản chọn lấy 200 đoạn theo cosine, giữ 80, kết hợp tối đa 80 tài liệu BM25 bằng RRF k=40 trọng số 0,5/0,5. Dense đối chứng luôn lấy 100 đoạn HNSW. Cùng một cách xếp hạng dùng cho benchmark (đọc 10 tài liệu đầu) và demo (đưa 5 tài liệu đầu vào kiểm tra bằng chứng).


In [ ]:
run([PY, "-m", "benchmarks.retrieval_study", "final", "--output", RUN])
retrieval = read("dev100.json")
assert all(retrieval["metrics"][key]["n"] == 100 for key in ("dense", "locked", "selected"))
print("Cấu hình chọn:", retrieval["metadata"]["selection"],
      "· ánh xạ cùng lượt:", retrieval["aliases_same_invocation"])
rows = []
for method, m in retrieval["metrics"].items():
    values = [m["recall"][f"@{k}"] for k in (1, 3, 5, 10)]
    values += [m["coverage"], m["candidate_union_coverage"], m["candidate50_document_coverage"]]
    rows.append([method, *(f"{v:.0%}" for v in values),
                 m["latency_ms"]["median"], m["latency_ms"]["p95"], m["errors"]])
table(["Cấu hình", "R@1", "R@3", "R@5", "R@10", "Gold trong index", "Hợp ứng viên",
       "Candidate50", "Median (ms)", "P95 (ms)", "Lỗi"], rows)


## 5. Ollama và Qwen3 trong runtime

Pha lập chỉ mục và đo truy xuất đã kết thúc. Ô này dùng lại hoặc khởi động một Ollama 0.40.17 ở loopback, kiểm tra tag `qwen3:4b-instruct` và SHA-256 của manifest trước khi sinh thử. Nếu thiếu Qwen, chỉ tag này được tải. Sai digest, sai phiên bản hay lỗi nạp model sẽ dừng lượt chạy. Số byte VRAM bên dưới do Ollama báo khi đã nạp model.


In [ ]:
ollama = json.loads(run([PY, "-m", "benchmarks.colab_runtime", "ollama", "--work", WORK],
                        capture=True).stdout)
write("ollama.json", ollama)
print(json.dumps(ollama, ensure_ascii=False, indent=2))


## 6. Chấm câu trả lời ở đúng `dev[:10]`

Mỗi phương án chạy toàn bộ `Assistant.ask` từ câu hỏi đến tài liệu, trích dẫn, quyết định trả lời hoặc từ chối. Qwen chấm cùng prompt và cùng 10 câu. Điểm judge, số từ chối và lỗi đều tính trên toàn bộ 10 câu. Judge dùng cùng model sinh, có thể chấm một câu từ chối là đúng. Recall là độ tìm thấy tài liệu và không đo độ đúng y khoa. README của bản nộp dẫn tới phần đối chiếu từng câu với nguồn trên Windows do agent thực hiện.


In [ ]:
generation_runs = [("dense", "Dense", []), ("selected", "Hybrid", [])]
if retrieval["metadata"]["selection"] != "locked":
    generation_runs.insert(1, ("locked", "Hybrid", ["--hybrid-backend", "locked"]))
else:
    print("Hybrid đầu kỳ và bản chọn có cùng cấu hình; dùng chung lượt selected.")
for label, method, opts in generation_runs:
    run([PY, "-m", "benchmarks.evaluate_generation", "--method", method, "--run-label", label,
         *opts, "--output", RUN / f"generation-{label}.json"])
rows = []
for label, _, _ in generation_runs:
    g = read(f"generation-{label}.json")
    assert g["n_questions"] == 10 and g["model_digest"].removeprefix("sha256:") == ollama["digest"]
    rows.append([label, f'{g["judge_hits"]}/10', f'{g["refusals"]}/10',
                 g["invalid_judgments_including_errors"], g["judge_request_errors"],
                 g["technical_failures"], round(g["latency_ms"]["ask"]["median"]),
                 round(g["latency_ms"]["ask"]["p95"]),
                 json.dumps(g["reason_counts"], ensure_ascii=False)])
table(["Cấu hình", "Judge", "Từ chối", "Judge không hợp lệ (gồm lỗi)", "Lỗi gọi judge",
       "Lỗi ứng dụng", "Median hỏi (ms)", "P95 hỏi (ms)", "Quyết định trong 10 câu"], rows)


## 7. Ví dụ và các đường lỗi

Sáu ca sử dụng câu hỏi tổng hợp. Kiểm tra dương tính yêu cầu câu trả lời nói đúng các yếu tố ảnh hưởng **kết quả đo lưu huyết não** và khớp nguồn. Một ca riêng kiểm tra hệ thống không dùng đoạn này để trả lời sai bệnh cảnh về nguyên nhân tăng huyết áp. Ca ngoài lĩnh vực hỏi thủ đô nước Úc; thuốc giả định kiểm tra thiếu bằng chứng. Ca cấp cứu và ca lỗi dịch vụ dùng endpoint loopback thử nghiệm riêng.


In [ ]:
from subprocess import CalledProcessError
try:
    run([PY, "-m", "benchmarks.check_demo", "--output", RUN / "demo.json"])
except CalledProcessError as error:
    if error.returncode != 1 or not (RUN / "demo.json").is_file():
        raise
demos = read("demo.json")

def show(answer, sources):
    print(answer if answer is not None else "(Văn bản ngoài mẫu công bố; cần đối chiếu trước khi chia sẻ.)")
    source_lines = []
    for item in sources:
        label = html.escape(f'[{item["number"]}] {item["doc_id"]} · {item["chunk_id"]}')
        url = item.get("url")
        if url:
            label = f'<a href="{html.escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">{label}</a>'
        label += " · đã trích dẫn" if item["cited"] else " · đã truy xuất"
        if item.get("excerpt"):
            label += "<blockquote>" + html.escape(item["excerpt"]) + "</blockquote>"
        source_lines.append(label)
    if source_lines:
        display(HTML("<br>".join(source_lines)))

for item in demos["cases"]:
    print(f'\n{item["question"]}\n{item["status"]} · {item["reason_code"]} · '
          f'{"PASS" if item["passed"] else "FAIL"}')
    show(item["answer"], item["sources"])
print("Kiểm tra demo:", "PASS" if demos["passed"] else "FAIL")


## 8. Hỏi ViCare

Nhập câu hỏi kiến thức sức khỏe chung vào ô dưới, rồi chạy lại riêng ô. Các liên kết nguồn đã được ứng dụng kiểm tra. Ô tương tác không ghi câu hỏi vào bộ kết quả. Colab có thể lưu nội dung cell và output cùng notebook; hãy xóa câu hỏi riêng trước khi chia sẻ.


In [ ]:
cau_hoi = ""  #@param {type:"string"}
if cau_hoi.strip():
    result = run([PY, "-c", """
import json, re, sys
from src.config import Settings
from src.generation import Assistant
from src.safety import validate_url
reply = Assistant(Settings.from_env()).ask(sys.stdin.read(), "Hybrid")
quoted = re.search("“(.+?)”", reply.answer)
sources = []
for number, hit in enumerate(reply.hits, 1):
    text = hit.chunk.text
    cited = str(number) in reply.citation_ids
    quote_at = text.find(quoted[1]) if cited and quoted else 0
    start = max(0, min(len(text) - 520, quote_at - 75))
    excerpt = ("…" if start else "") + text[start:start + 520] + ("…" if start + 520 < len(text) else "")
    sources.append({"number": number, "doc_id": hit.chunk.doc_id, "chunk_id": hit.chunk.chunk_id,
                    "url": validate_url(hit.chunk.url), "cited": cited, "excerpt": excerpt})
print(json.dumps({"answer": reply.answer, "status": reply.status, "reason": reply.reason_code,
                  "sources": sources}, ensure_ascii=False))
"""], input=cau_hoi, capture=True)
    reply = json.loads(result.stdout)
    print(reply["status"], "·", reply["reason"])
    show(reply["answer"], reply["sources"])
else:
    print("Nhập câu hỏi ở cau_hoi và chạy riêng ô này.")


## 9. Kết luận và xuất kết quả

So sánh Recall@5 cùng các mốc khác và cân nhắc độ trễ, tỷ lệ từ chối, lỗi, bằng chứng của từng ca. Chỉ số Judge của Qwen cần được đọc cùng bảng quyết định và đối chiếu nguồn. Ngưỡng cosine là ngưỡng demo chưa được thẩm định lâm sàng. Chỉ mục HNSW dựng mới và cách Ollama sử dụng phần cứng có thể làm kết quả khác máy tác giả.

File ZIP gồm JSON môi trường, phiên bản, corpus và chỉ mục, chỉ số, từng câu truy xuất, thống kê generation và các ca demo tổng hợp. CSV chỉ lưu ID và kết quả đánh giá. Tải ZIP trong khung **Files** của Colab, rồi chọn **File → Download → Download .ipynb** để lưu các cell có output trước khi kết thúc runtime.


In [ ]:
per_question = [json.loads(line) for line in (RUN / "dev100.jsonl").read_text(encoding="utf-8").splitlines()]
def export_csv(name, columns, records):
    with (RUN / name).open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
export_csv("retrieval.csv", ["id", "method", "rank", "dense_rank", "bm25_rank", "rrf_rank",
                             "union_hit", "candidate50_hit", "latency_ms"], per_question)
gen_rows = [dict(row, configuration=label) for label, _, _ in generation_runs
            for row in read(f"generation-{label}.json")["rows"]]
export_csv("generation.csv", ["configuration", "id", "status", "reason_code", "judge_score",
                              "judge_error_type", "generation_ms", "judge_ms"], gen_rows)
selected = retrieval["metrics"]["selected"]
base = retrieval["metrics"]["dense"]
print(f'Dev100: Recall@5 Dense {base["recall"]["@5"]:.0%}, bản chọn {selected["recall"]["@5"]:.0%}.')
print("Ca demo và an toàn:", "PASS" if demos["passed"] else "FAIL · xem chi tiết demo.json")
filenames = ["environment.json", "versions.json", "regression.json", "corpus.json", "ollama.json",
             "dev100.json", "dev100.jsonl", "retrieval.csv", "generation.csv", "demo.json",
             *(f"generation-{label}.json" for label, _, _ in generation_runs)]
archive = RUN.parent / (RUN.name + ".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in filenames:
        bundle.write(RUN / name, name)
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None and set(bundle.namelist()) == set(filenames)
print("ZIP kết quả:", archive)


### Nguồn và điều kiện sử dụng

- Dữ liệu ViHealthQA của Nguyen và cộng sự, revision `d90a62deedfc822a15de3f35535bb41474e4d3f4`. Tác giả dành dữ liệu cho nghiên cứu và cảnh báo về thông tin cá nhân. [Dataset card](https://huggingface.co/datasets/tarudesu/ViHealthQA) · [SPBERTQA](https://arxiv.org/abs/2206.09600)
- Embedding `intfloat/multilingual-e5-small`, MIT, revision `614241f622f53c4eeff9890bdc4f31cfecc418b3`. [E5 model card](https://huggingface.co/intfloat/multilingual-e5-small)
- Qwen3-4B-Instruct, Apache-2.0, phục vụ qua Ollama. [Model](https://ollama.com/library/qwen3:4b-instruct) · [Ollama chat API](https://docs.ollama.com/api/chat)
- Thiết kế known-corpus và Dev100 kế thừa notebook thực hành của BTC. Mã ứng dụng và kiểm thử có sử dụng công cụ hỗ trợ lập trình AI. [Source đã ghim](https://github.com/Vdbinh05/ViCare-AI-Challenge/tree/b9a586aecf9837fdf991dddd55f07d26b43d3b64) · [Giới hạn Colab](https://research.google.com/colaboratory/faq.html)
